<a href="https://colab.research.google.com/github/kdkim2000/RAG2026/blob/main/%5B%EA%B0%95%EC%9D%98%EB%82%B4%EC%9A%A9%5D_7_LangChain%EC%9D%84_%EC%9D%B4%EC%9A%A9%ED%95%9C_SQL_%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%B2%A0%EC%9D%B4%EC%8A%A4_%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [실습] LangChain을 이용한 SQL 데이터베이스 분석

Retrieval과 함께, LLM이 외부 문서를 참조하는 방법인 Text-to-SQL에 대해 알아보겠습니다.   

LLM에게 DB 스키마와 질문을 전달하여, 쿼리를 만들고 실행합니다.

In [ ]:
!pip install openai langchain langchain-community langchain-openai matplotlib dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 1.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv(override=True)


llm = ChatOpenAI(model = 'gpt-5.6-luna', reasoning_effort='low')
llm.invoke('안녕')

AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 8, 'total_tokens': 22, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-E8zOuuEOuELqNE3TN3IdRyRuSwlJ7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fcaa6-c6c6-75b0-bbf3-25d0c50e1119-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 14, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 1. 예제 SQL 파일 만들기   

가상의 도서관 데이터베이스를 만들어 활용하겠습니다.   
책, 유저, 대여 정보로 구성된 3개의 테이블을 사용하겠습니다.

In [ ]:
import sqlite3
from datetime import datetime, timedelta

# 현재 날짜 기준 설정
today = datetime.now().date()

# SQLite DB 파일 연결
conn = sqlite3.connect('library.db')
c = conn.cursor()

# 1. 출판사 테이블
c.execute('''
CREATE TABLE IF NOT EXISTS publishers (
    publisher_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    country TEXT,
    established_year INTEGER
)
''')

# 2. 저자 테이블 (다대다 관계를 위해 분리)
c.execute('''
CREATE TABLE IF NOT EXISTS authors (
    author_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    birth_year INTEGER,
    nationality TEXT
)
''')

# 3. 카테고리 테이블 (정규화)
c.execute('''
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    description TEXT
)
''')

# 4. 책 테이블 (출판사, 카테고리 외래키 추가)
c.execute('''
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    isbn TEXT UNIQUE,
    publication_year INTEGER,
    publisher_id INTEGER,
    category_id INTEGER,
    page_count INTEGER,
    language TEXT DEFAULT 'Korean',
    FOREIGN KEY(publisher_id) REFERENCES publishers(publisher_id),
    FOREIGN KEY(category_id) REFERENCES categories(category_id)
)
''')

# 5. 책-저자 연결 테이블 (다대다 관계)
c.execute('''
CREATE TABLE IF NOT EXISTS book_authors (
    book_id INTEGER,
    author_id INTEGER,
    author_order INTEGER,
    PRIMARY KEY(book_id, author_id),
    FOREIGN KEY(book_id) REFERENCES books(book_id),
    FOREIGN KEY(author_id) REFERENCES authors(author_id)
)
''')

# 6. 책 복본 테이블 (실제 물리적 책)
c.execute('''
CREATE TABLE IF NOT EXISTS book_copies (
    copy_id INTEGER PRIMARY KEY,
    book_id INTEGER,
    barcode TEXT UNIQUE,
    condition TEXT CHECK(condition IN ('excellent', 'good', 'fair', 'poor')),
    location TEXT,
    acquisition_date TEXT,
    status TEXT DEFAULT 'available' CHECK(status IN ('available', 'borrowed', 'lost', 'repair')),
    FOREIGN KEY(book_id) REFERENCES books(book_id)
)
''')

# 7. 회원 등급 테이블
c.execute('''
CREATE TABLE IF NOT EXISTS membership_types (
    type_id INTEGER PRIMARY KEY,
    type_name TEXT NOT NULL,
    max_books INTEGER,
    loan_period_days INTEGER,
    late_fee_per_day REAL
)
''')

# 8. 회원 테이블 (등급 추가)
c.execute('''
CREATE TABLE IF NOT EXISTS members (
    member_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE,
    phone TEXT,
    address TEXT,
    join_date TEXT,
    membership_type_id INTEGER DEFAULT 1,
    is_active INTEGER DEFAULT 1,
    FOREIGN KEY(membership_type_id) REFERENCES membership_types(type_id)
)
''')

# 9. 대여 정보 테이블 (실제 복본 기준)
c.execute('''
CREATE TABLE IF NOT EXISTS rentals (
    rental_id INTEGER PRIMARY KEY,
    copy_id INTEGER,
    member_id INTEGER,
    rental_date TEXT,
    due_date TEXT,
    return_date TEXT,
    status TEXT DEFAULT 'active' CHECK(status IN ('active', 'returned', 'overdue')),
    FOREIGN KEY(copy_id) REFERENCES book_copies(copy_id),
    FOREIGN KEY(member_id) REFERENCES members(member_id)
)
''')

# 10. 예약 테이블
c.execute('''
CREATE TABLE IF NOT EXISTS reservations (
    reservation_id INTEGER PRIMARY KEY,
    book_id INTEGER,
    member_id INTEGER,
    reservation_date TEXT,
    status TEXT DEFAULT 'waiting' CHECK(status IN ('waiting', 'ready', 'cancelled', 'completed')),
    expiry_date TEXT,
    FOREIGN KEY(book_id) REFERENCES books(book_id),
    FOREIGN KEY(member_id) REFERENCES members(member_id)
)
''')

# 11. 연체료 테이블
c.execute('''
CREATE TABLE IF NOT EXISTS late_fees (
    fee_id INTEGER PRIMARY KEY,
    rental_id INTEGER,
    amount REAL,
    calculation_date TEXT,
    paid INTEGER DEFAULT 0,
    payment_date TEXT,
    FOREIGN KEY(rental_id) REFERENCES rentals(rental_id)
)
''')

# 12. 리뷰 테이블
c.execute('''
CREATE TABLE IF NOT EXISTS reviews (
    review_id INTEGER PRIMARY KEY,
    book_id INTEGER,
    member_id INTEGER,
    rating INTEGER CHECK(rating >= 1 AND rating <= 5),
    comment TEXT,
    review_date TEXT,
    FOREIGN KEY(book_id) REFERENCES books(book_id),
    FOREIGN KEY(member_id) REFERENCES members(member_id)
)
''')

print("테이블 생성 완료!")

# === 데이터 삽입 ===

# 1. 출판사 데이터
publishers_data = [
    ('문학동네', '대한민국', 1993),
    ('민음사', '대한민국', 1966),
    ('Bloomsbury', '영국', 1986),
    ('Scribner', '미국', 1846),
    ('Penguin Books', '영국', 1935)
]
c.executemany("INSERT INTO publishers (name, country, established_year) VALUES (?, ?, ?)", publishers_data)

# 2. 카테고리 데이터
categories_data = [
    ('판타지', '마법, 신화, 초자연적 요소가 포함된 소설'),
    ('고전문학', '시대를 초월한 문학적 가치를 지닌 작품'),
    ('SF', '과학과 기술을 기반으로 한 공상과학 소설'),
    ('소설', '일반 문학 소설'),
    ('디스토피아', '암울한 미래 사회를 다룬 작품'),
    ('동화', '어린이를 위한 이야기')
]
c.executemany("INSERT INTO categories (name, description) VALUES (?, ?)", categories_data)

# 3. 저자 데이터
authors_data = [
    ('J.K. 롤링', 1965, '영국'),
    ('어니스트 헤밍웨이', 1899, '미국'),
    ('김초엽', 1993, '대한민국'),
    ('헤르만 헤세', 1877, '독일'),
    ('J.R.R. 톨킨', 1892, '영국'),
    ('조지 오웰', 1903, '영국'),
    ('루이스 캐럴', 1832, '영국'),
    ('F. 스콧 피츠제럴드', 1896, '미국'),
    ('이민진', 1968, '대한민국'),
    ('J.D. Salinger', 1919, '미국'),
    ('Aldous Huxley', 1894, '영국')
]
c.executemany("INSERT INTO authors (name, birth_year, nationality) VALUES (?, ?, ?)", authors_data)

# 4. 책 데이터
books_data = [
    ('해리 포터와 마법사의 돌', '9788983920003', 1997, 3, 1, 320, 'Korean'),
    ('노인과 바다', '9788937460012', 1952, 4, 2, 128, 'Korean'),
    ('지구 끝의 온실', '9791164371273', 2016, 1, 3, 256, 'Korean'),
    ('데미안', '9788937462788', 1919, 2, 4, 224, 'Korean'),
    ('반지의 제왕', '9788937460893', 1954, 2, 1, 576, 'Korean'),
    ('1984', '9788937462672', 1949, 5, 5, 400, 'Korean'),
    ('이상한 나라의 앨리스', '9788937460821', 1865, 5, 6, 192, 'Korean'),
    ('위대한 개츠비', '9788937462627', 1925, 4, 4, 256, 'Korean'),
    ('파친코', '9788932473536', 2017, 1, 4, 640, 'Korean'),
    ('The Catcher in the Rye', '9780316769174', 1951, 4, 4, 288, 'English'),
    ('Brave New World', '9780060850524', 1932, 5, 5, 311, 'English')
]
c.executemany("""INSERT INTO books (title, isbn, publication_year, publisher_id, category_id,
              page_count, language) VALUES (?, ?, ?, ?, ?, ?, ?)""", books_data)

# 5. 책-저자 연결
book_authors_data = [
    (1, 1, 1), (2, 2, 1), (3, 3, 1), (4, 4, 1),
    (5, 5, 1), (6, 6, 1), (7, 7, 1), (8, 8, 1),
    (9, 9, 1), (10, 10, 1), (11, 11, 1)
]
c.executemany("INSERT INTO book_authors (book_id, author_id, author_order) VALUES (?, ?, ?)", book_authors_data)

# 6. 책 복본 생성 (각 책당 2-3권, 현재 날짜 기준으로 입고일 설정)
book_copies_data = []
for book_id in range(1, 12):
    num_copies = 2 if book_id % 2 == 0 else 3
    for i in range(num_copies):
        barcode = f"BC{book_id:03d}{i+1:02d}"
        condition = 'excellent' if i == 0 else 'good'
        location = f"A{(book_id-1)//3 + 1}-{(book_id-1)%3 + 1}"
        # 입고일을 6개월 전부터 최근까지 다양하게 설정
        acquisition_days_ago = 180 - (book_id * 10) - (i * 5)
        acquisition_date = (today - timedelta(days=acquisition_days_ago)).isoformat()
        status = 'available' if i > 0 else 'borrowed'
        book_copies_data.append((book_id, barcode, condition, location, acquisition_date, status))

c.executemany("""INSERT INTO book_copies (book_id, barcode, condition, location, acquisition_date, status)
              VALUES (?, ?, ?, ?, ?, ?)""", book_copies_data)

# 7. 회원 등급
membership_types_data = [
    ('일반', 3, 14, 100),
    ('우수', 5, 21, 50),
    ('VIP', 10, 30, 0)
]
c.executemany("""INSERT INTO membership_types (type_name, max_books, loan_period_days, late_fee_per_day)
              VALUES (?, ?, ?, ?)""", membership_types_data)

# 8. 회원 데이터 (가입일을 최근 날짜로 설정)
members_data = [
    ('김민준', 'minjun@example.com', '010-1234-5678', '서울시 강남구',
     (today - timedelta(days=60)).isoformat(), 1, 1),
    ('이하늘', 'haneul@example.com', '010-2345-6789', '서울시 서초구',
     (today - timedelta(days=55)).isoformat(), 2, 1),
    ('박서준', 'seojoon@example.com', '010-3456-7890', '서울시 송파구',
     (today - timedelta(days=50)).isoformat(), 1, 1),
    ('최유리', 'yuri@example.com', '010-4567-8901', '서울시 마포구',
     (today - timedelta(days=45)).isoformat(), 3, 1),
    ('정다은', 'daeun@example.com', '010-5678-9012', '서울시 용산구',
     (today - timedelta(days=40)).isoformat(), 2, 1)
]
c.executemany("""INSERT INTO members (name, email, phone, address, join_date, membership_type_id, is_active)
              VALUES (?, ?, ?, ?, ?, ?, ?)""", members_data)

# 9. 대여 정보 (현재 날짜 기준으로 다양한 상황 생성)
rentals_data = [
    # 정상 대여중 (반납일이 미래)
    (1, 1, (today - timedelta(days=10)).isoformat(),
           (today + timedelta(days=4)).isoformat(), None, 'active'),

    # 연체중 (반납일 지남, 아직 반납 안함) - 5일 연체
    (4, 2, (today - timedelta(days=26)).isoformat(),
           (today - timedelta(days=5)).isoformat(), None, 'overdue'),

    # 연체 후 반납 완료 (2일 연체했다가 반납)
    (7, 3, (today - timedelta(days=20)).isoformat(),
           (today - timedelta(days=10)).isoformat(),
           (today - timedelta(days=8)).isoformat(), 'returned'),

    # VIP 정상 대여중
    (10, 4, (today - timedelta(days=5)).isoformat(),
            (today + timedelta(days=25)).isoformat(), None, 'active'),

    # 정상 반납 완료
    (13, 5, (today - timedelta(days=15)).isoformat(),
            (today - timedelta(days=1)).isoformat(),
            (today - timedelta(days=2)).isoformat(), 'returned'),

    # 연체중 (3일 연체)
    (16, 1, (today - timedelta(days=17)).isoformat(),
            (today - timedelta(days=3)).isoformat(), None, 'overdue'),
]
c.executemany("""INSERT INTO rentals (copy_id, member_id, rental_date, due_date, return_date, status)
              VALUES (?, ?, ?, ?, ?, ?)""", rentals_data)

# 10. 예약 정보 (현재 날짜 기준)
reservations_data = [
    (1, 3, (today - timedelta(days=5)).isoformat(), 'waiting',
     (today + timedelta(days=9)).isoformat()),
    (5, 1, (today - timedelta(days=3)).isoformat(), 'waiting',
     (today + timedelta(days=11)).isoformat())
]
c.executemany("""INSERT INTO reservations (book_id, member_id, reservation_date, status, expiry_date)
              VALUES (?, ?, ?, ?, ?)""", reservations_data)

# 11. 리뷰 데이터 (현재 날짜 기준 최근 리뷰들)
reviews_data = [
    (2, 2, 5, '정말 감동적인 작품이었습니다. 인생 책으로 추천합니다!',
     (today - timedelta(days=10)).isoformat()),
    (3, 5, 4, 'SF의 새로운 가능성을 보여준 소설. 추천해요.',
     (today - timedelta(days=5)).isoformat()),
    (1, 1, 5, '어린 시절 읽었던 기억이 새롭습니다. 명작!',
     (today - timedelta(days=15)).isoformat()),
    (5, 3, 5, '판타지의 정석! 웅장한 세계관에 빠져들었습니다.',
     (today - timedelta(days=12)).isoformat()),
    (6, 4, 5, '디스토피아 소설의 걸작. 현대 사회를 되돌아보게 만드네요.',
     (today - timedelta(days=8)).isoformat()),
    (4, 2, 4, '자아 성찰에 대한 깊은 통찰이 담긴 책. 청소년기에 꼭 읽어야 할 소설입니다.',
     (today - timedelta(days=7)).isoformat()),
    (8, 1, 4, '1920년대 미국 사회의 화려함과 공허함을 잘 표현했어요.',
     (today - timedelta(days=11)).isoformat()),
    (9, 5, 5, '재일교포의 역사를 통해 인간의 존엄성을 느꼈습니다. 강추!',
     (today - timedelta(days=3)).isoformat()),
    (7, 3, 3, '동화지만 철학적 의미가 깊네요. 어른이 읽어도 좋습니다.',
     (today - timedelta(days=9)).isoformat()),
    (10, 4, 4, '10대의 방황과 고민이 생생하게 그려진 작품. 공감 100%',
     (today - timedelta(days=6)).isoformat()),

    # 별로인 리뷰 (1-2점)
    (11, 1, 2, '너무 암울하고 우울해요. 기대했는데 실망스럽습니다.',
     (today - timedelta(days=4)).isoformat()),
    (6, 3, 1, '지루하고 난해합니다. 끝까지 읽기 힘들었어요.',
     (today - timedelta(days=2)).isoformat()),
    (4, 5, 2, '추상적인 내용이 많아서 이해하기 어렵네요. 별로 공감이 안 갑니다.',
     (today - timedelta(days=1)).isoformat()),
    (1, 4, 1, '과대평가된 책 같아요. 스토리가 예상 가능하고 유치합니다.',
     today.isoformat())
]
c.executemany("""INSERT INTO reviews (book_id, member_id, rating, comment, review_date)
              VALUES (?, ?, ?, ?, ?)""", reviews_data)

conn.commit()
print("\n데이터 삽입 완료!")

# === 연체료 자동 계산 ===
print("\n=== 연체료 계산 중... ===\n")

def calculate_late_fees(cursor):
    """연체료 자동 계산 및 삽입"""

    # 연체중인 대여 건 조회 (날짜만 비교)
    cursor.execute("""
        SELECT
            r.rental_id,
            DATE(r.due_date) as due_date,
            DATE(r.return_date) as return_date,
            mt.late_fee_per_day,
            m.name
        FROM rentals r
        JOIN members m ON r.member_id = m.member_id
        JOIN membership_types mt ON m.membership_type_id = mt.type_id
        WHERE (r.return_date IS NULL AND DATE(r.due_date) < DATE('now'))
           OR (r.return_date IS NOT NULL AND DATE(r.return_date) > DATE(r.due_date))
    """)

    for rental_id, due_date, return_date, fee_per_day, member_name in cursor.fetchall():
        # 연체 일수 계산 (SQLite의 날짜 함수 사용하여 일관성 유지)
        cursor.execute("""
            SELECT CAST((JULIANDAY(?) - JULIANDAY(?)) AS INTEGER)
        """, (return_date if return_date else datetime.now().date().isoformat(), due_date))

        overdue_days = cursor.fetchone()[0]

        if overdue_days > 0:
            amount = overdue_days * fee_per_day
            paid_status = 1 if return_date else 0  # 반납했으면 납부완료

            # 이미 연체료가 있는지 확인
            cursor.execute("SELECT fee_id FROM late_fees WHERE rental_id = ?", (rental_id,))
            if not cursor.fetchone():
                payment_date = return_date if paid_status else None
                cursor.execute("""
                    INSERT INTO late_fees (rental_id, amount, calculation_date, paid, payment_date)
                    VALUES (?, ?, DATE('now'), ?, ?)
                """, (rental_id, amount, paid_status, payment_date))
                status = "납부완료" if paid_status else "미납"
                print(f"   {member_name}: {overdue_days}일 연체, {amount:,.0f}원 ({status})")

calculate_late_fees(c)
conn.commit()

# === 복잡한 쿼리 예제 ===
print("\n=== 쿼리 예제 ===\n")

# 1. 책별 평균 평점과 리뷰 수
print("1. 책별 평균 평점:")
c.execute("""
    SELECT b.title, AVG(r.rating) as avg_rating, COUNT(r.review_id) as review_count
    FROM books b
    LEFT JOIN reviews r ON b.book_id = r.book_id
    GROUP BY b.book_id
    HAVING review_count > 0
    ORDER BY avg_rating DESC
""")
for row in c.fetchall():
    print(f"   {row[0]}: {row[1]:.1f}점 ({row[2]}개 리뷰)")

# 2. 회원별 대여 현황 및 연체 정보
print("\n2. 회원별 대여 현황:")
c.execute("""
    SELECT m.name, mt.type_name,
           COUNT(CASE WHEN r.status IN ('active', 'overdue') THEN 1 END) as active_rentals,
           mt.max_books as max_allowed,
           SUM(CASE WHEN lf.paid = 0 THEN lf.amount ELSE 0 END) as unpaid_fees
    FROM members m
    JOIN membership_types mt ON m.membership_type_id = mt.type_id
    LEFT JOIN rentals r ON m.member_id = r.member_id
    LEFT JOIN late_fees lf ON r.rental_id = lf.rental_id
    GROUP BY m.member_id
""")
for row in c.fetchall():
    print(f"   {row[0]} ({row[1]}): {row[2]}/{row[3]}권 대여중, 미납 연체료: {row[4] if row[4] else 0:,.0f}원")

# 3. 가장 인기있는 책 (대여 횟수 기준)
print("\n3. 인기 도서 TOP 5:")
c.execute("""
    SELECT b.title, a.name as author, COUNT(r.rental_id) as rental_count
    FROM books b
    JOIN book_authors ba ON b.book_id = ba.book_id
    JOIN authors a ON ba.author_id = a.author_id
    JOIN book_copies bc ON b.book_id = bc.book_id
    LEFT JOIN rentals r ON bc.copy_id = r.copy_id
    GROUP BY b.book_id
    ORDER BY rental_count DESC
    LIMIT 5
""")
for idx, row in enumerate(c.fetchall(), 1):
    print(f"   {idx}. {row[0]} - {row[1]} ({row[2]}회)")

# 4. 현재 대여 가능한 책 목록
print("\n4. 현재 대여 가능한 책:")
c.execute("""
    SELECT DISTINCT b.title, a.name as author, COUNT(bc.copy_id) as available_copies
    FROM books b
    JOIN book_authors ba ON b.book_id = ba.book_id
    JOIN authors a ON ba.author_id = a.author_id
    JOIN book_copies bc ON b.book_id = bc.book_id
    WHERE bc.status = 'available'
    GROUP BY b.book_id
    ORDER BY available_copies DESC
""")
for row in c.fetchall():
    print(f"   {row[0]} - {row[1]} ({row[2]}권 가능)")

# 5. 현재 연체중인 도서 상세
print("\n5. 현재 연체중인 도서:")
c.execute("""
    SELECT
        m.name,
        b.title,
        DATE(r.due_date) as due_date,
        CAST((JULIANDAY(DATE('now')) - JULIANDAY(DATE(r.due_date))) AS INTEGER) as overdue_days,
        CAST((JULIANDAY(DATE('now')) - JULIANDAY(DATE(r.due_date))) AS INTEGER) * mt.late_fee_per_day as fee
    FROM rentals r
    JOIN members m ON r.member_id = m.member_id
    JOIN membership_types mt ON m.membership_type_id = mt.type_id
    JOIN book_copies bc ON r.copy_id = bc.copy_id
    JOIN books b ON bc.book_id = b.book_id
    WHERE r.return_date IS NULL AND DATE(r.due_date) < DATE('now')
    ORDER BY overdue_days DESC
""")
overdue_list = c.fetchall()
if overdue_list:
    for row in overdue_list:
        print(f"   {row[0]}: {row[1]} (연체 {row[3]}일, 연체료 {row[4]:,}원)")
else:
    print("   연체중인 도서가 없습니다.")

conn.close()
print("\n데이터베이스 연결 종료!")
print(f"\n✅ 모든 날짜가 {today} 기준으로 설정되었습니다.")

테이블 생성 완료!

데이터 삽입 완료!

=== 연체료 계산 중... ===

   이하늘: 5일 연체, 250원 (미납)
   박서준: 2일 연체, 200원 (납부완료)
   김민준: 3일 연체, 300원 (미납)

=== 쿼리 예제 ===

1. 책별 평균 평점:
   노인과 바다: 5.0점 (1개 리뷰)
   반지의 제왕: 5.0점 (1개 리뷰)
   파친코: 5.0점 (1개 리뷰)
   지구 끝의 온실: 4.0점 (1개 리뷰)
   위대한 개츠비: 4.0점 (1개 리뷰)
   The Catcher in the Rye: 4.0점 (1개 리뷰)
   해리 포터와 마법사의 돌: 3.0점 (2개 리뷰)
   데미안: 3.0점 (2개 리뷰)
   1984: 3.0점 (2개 리뷰)
   이상한 나라의 앨리스: 3.0점 (1개 리뷰)
   Brave New World: 2.0점 (1개 리뷰)

2. 회원별 대여 현황:
   김민준 (일반): 2/3권 대여중, 미납 연체료: 300원
   이하늘 (우수): 1/5권 대여중, 미납 연체료: 250원
   박서준 (일반): 0/3권 대여중, 미납 연체료: 0원
   최유리 (VIP): 1/10권 대여중, 미납 연체료: 0원
   정다은 (우수): 0/5권 대여중, 미납 연체료: 0원

3. 인기 도서 TOP 5:
   1. 해리 포터와 마법사의 돌 - J.K. 롤링 (1회)
   2. 노인과 바다 - 어니스트 헤밍웨이 (1회)
   3. 지구 끝의 온실 - 김초엽 (1회)
   4. 데미안 - 헤르만 헤세 (1회)
   5. 반지의 제왕 - J.R.R. 톨킨 (1회)

4. 현재 대여 가능한 책:
   Brave New World - Aldous Huxley (2권 가능)
   파친코 - 이민진 (2권 가능)
   이상한 나라의 앨리스 - 루이스 캐럴 (2권 가능)
   반지의 제왕 - J.R.R. 톨킨 (2권 가능)
   지구 끝의 온실 - 김초엽 (2권 가능)
   해리 포터와 마법사의 돌 - J.K. 롤링 (2권 

## 2. `create_sql_query_chain `

이제, LangChain을 이용해 SQL DB를 처리하는 방법에 대해 알아보겠습니다.

langchain의 부가기능 중 SQLDatabase 를 이용해 db를 불러옵니다.

In [ ]:
from langchain_classic.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///library.db")

db의 `get_table_info()` 결과를 프롬프트에 전달하여 분석을 수행합니다.

In [ ]:
print(db.get_table_info())


CREATE TABLE authors (
	author_id INTEGER, 
	name TEXT NOT NULL, 
	birth_year INTEGER, 
	nationality TEXT, 
	PRIMARY KEY (author_id)
)

/*
3 rows from authors table:
author_id	name	birth_year	nationality
1	J.K. 롤링	1965	영국
2	어니스트 헤밍웨이	1899	미국
3	김초엽	1993	대한민국
*/


CREATE TABLE book_authors (
	book_id INTEGER, 
	author_id INTEGER, 
	author_order INTEGER, 
	PRIMARY KEY (book_id, author_id), 
	FOREIGN KEY(book_id) REFERENCES books (book_id), 
	FOREIGN KEY(author_id) REFERENCES authors (author_id)
)

/*
3 rows from book_authors table:
book_id	author_id	author_order
1	1	1
2	2	1
3	3	1
*/


CREATE TABLE book_copies (
	copy_id INTEGER, 
	book_id INTEGER, 
	barcode TEXT, 
	condition TEXT, 
	location TEXT, 
	acquisition_date TEXT, 
	status TEXT DEFAULT 'available', 
	PRIMARY KEY (copy_id), 
	FOREIGN KEY(book_id) REFERENCES books (book_id), 
	UNIQUE (barcode), 
	CHECK (condition IN ('excellent', 'good', 'fair', 'poor')), 
	CHECK (status IN ('available', 'borrowed', 'lost', 'repair'))
)

/*
3 rows 

`create_sql_query_chain`은 질문에 대한 SQL 쿼리를 생성합니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableParallel

sql_prompt = ChatPromptTemplate([
    ('system','''SQL 테이블의 정보와 질문이 주어집니다.
테이블을 참고하여, 질문을 쿼리로 변환하세요.
반드시 하나의 쿼리만 작성하고, 여러 개의 쿼리를 세미콜론으로 구분하지 마세요.
만약 유저가 데이터의 개수를 지정하지 않는다면, {top_k} 개만 출력하세요.
SQLite의 경우 LIMIT clause 를 사용하면 됩니다.
데이터의 결과를 더 잘 전달하기 위해 정렬할 수 있습니다.
주어진 예시 row에 없는 내용이라도, 쿼리를 작성하세요.

정답과 무관한 컬럼에 대한 쿼리를 수행하지 말고, 필요한 내용만 추출하세요.
아래에 주어지는 컬럼 이름과 테이블 정보를 꼭 참고하세요.
존재하지 않는 컬럼에 대한 쿼리를 하지 않도록 주의하세요.
만약 '오늘'과 같이 현재 날짜에 대한 질문이 들어오면 date() 등을 사용하세요.
컬럼 이름은 "로 구분하고, 전체 출력은 마크다운으로 수행하세요.'''),
    ('human','''
테이블 정보: {table_info}
질문: {input}
---
SQL Query:''')
])


def _create_sql_query_chain(llm, db, prompt, k = 5):

    sql_chain = (
        prompt.partial(table_info = db.get_table_info(), top_k = str(k))
        # 입력 변수 table_info, top_k 미리 추가
        | llm

        | StrOutputParser()
        | (lambda x: x.strip())
        # 공백제거
    )
    return sql_chain

In [ ]:
query_chain = _create_sql_query_chain(llm,
                                     db,
                                     prompt = sql_prompt)

In [ ]:
response = query_chain.invoke("해리 포터와 마법사의 돌을 지금 빌릴 수 있나요?")

print(response)

```sql
SELECT
    b."title",
    COUNT(CASE WHEN bc."status" = 'available' THEN 1 END) AS "available_copies",
    CASE
        WHEN COUNT(CASE WHEN bc."status" = 'available' THEN 1 END) > 0 THEN '예'
        ELSE '아니오'
    END AS "can_borrow"
FROM "books" AS b
LEFT JOIN "book_copies" AS bc
    ON b."book_id" = bc."book_id"
WHERE b."title" = '해리 포터와 마법사의 돌'
GROUP BY b."book_id", b."title"
LIMIT 5
```


In [ ]:
# response의 마크다운 형식 처리하기

if "```" in response:
    response = response.split('```sql\n')[1].split('\n```')[0]
print(response)

SELECT
    b."title",
    COUNT(CASE WHEN bc."status" = 'available' THEN 1 END) AS "available_copies",
    CASE
        WHEN COUNT(CASE WHEN bc."status" = 'available' THEN 1 END) > 0 THEN '예'
        ELSE '아니오'
    END AS "can_borrow"
FROM "books" AS b
LEFT JOIN "book_copies" AS bc
    ON b."book_id" = bc."book_id"
WHERE b."title" = '해리 포터와 마법사의 돌'
GROUP BY b."book_id", b."title"
LIMIT 5


제대로 작동하는지 확인해 보겠습니다.

In [ ]:
db.run(response)

"[('해리 포터와 마법사의 돌', 2, '예')]"

## 쿼리 실행하기

`QuerySQLDatabaseTool`은 쿼리를 받아 실행합니다.

In [ ]:
from langchain_community.tools import QuerySQLDatabaseTool

execute_query = QuerySQLDatabaseTool(db = db)

execute_query.invoke(response)

"[('해리 포터와 마법사의 돌', 2, '예')]"

두 체인을 연결해 보겠습니다. 중간 파싱 함수가 필요합니다.

In [ ]:
# ```sql, ``` 사이의 문자열만 파싱
def parse_sql(response):
    if "```" in response:
        response = response.split('```sql\n')[1].split('\n```')[0]
    return response

chain = query_chain | parse_sql | execute_query
chain.invoke("해리 포터와 마법사의 돌을 지금 빌릴 수 있나요?")

"[('해리 포터와 마법사의 돌', '대출 가능', 2)]"

임의의 SQL 쿼리를 실행해, 값을 삭제하거나 추가할 수도 있습니다.

In [ ]:
chain.invoke("회원은 누가 있나요?")

"[(1, '김민준', 'minjun@example.com', '010-1234-5678', '서울시 강남구', '2026-06-05', 1, 1), (2, '이하늘', 'haneul@example.com', '010-2345-6789', '서울시 서초구', '2026-06-10', 2, 1), (3, '박서준', 'seojoon@example.com', '010-3456-7890', '서울시 송파구', '2026-06-15', 1, 1), (4, '최유리', 'yuri@example.com', '010-4567-8901', '서울시 마포구', '2026-06-20', 3, 1), (5, '정다은', 'daeun@example.com', '010-5678-9012', '서울시 용산구', '2026-06-25', 2, 1)]"

In [ ]:
chain.invoke("박서준 회원 삭제해줘")

''

In [ ]:
chain.invoke("회원은 누가 있나요?")

"[(1, '김민준', 'minjun@example.com', '010-1234-5678', '서울시 강남구', '2026-06-05', 1, 1), (2, '이하늘', 'haneul@example.com', '010-2345-6789', '서울시 서초구', '2026-06-10', 2, 1), (4, '최유리', 'yuri@example.com', '010-4567-8901', '서울시 마포구', '2026-06-20', 3, 1), (5, '정다은', 'daeun@example.com', '010-5678-9012', '서울시 용산구', '2026-06-25', 2, 1)]"

In [ ]:
chain.invoke("삼성SDS RAG 심화과정 교재 100권 추가해줘")

''

In [ ]:
chain.invoke("가장 재고가 많은 책이 뭐야?")

"[('해리 포터와 마법사의 돌', 3), ('지구 끝의 온실', 3), ('반지의 제왕', 3), ('이상한 나라의 앨리스', 3), ('파친코', 3)]"

<br><br>
## 쿼리 생성 후 응답하기
질문이 들어오면, 쿼리를 받아 응답합니다.

In [ ]:
answer_prompt = ChatPromptTemplate([
('system','''
당신은 매우 활발하고 유머러스한 도서관 사서 AI입니다.
모든 대화는 책 속 유명한 인물의 말투를 그대로 과장되게 따라하세요.
맨 뒤에, 괄호를 통해 누구인지 알려주세요. (OOO 톤으로)

질문과 SQL 쿼리, 쿼리의 실행 결과가 주어집니다.
해당 정보를 바탕으로 질문에 대한 답변을 생성하세요.
만약 결과가 없는 경우, 질문의 내용을 고려하여 적절한 답변을 생성하세요.
대출 조회 결과가 없는 경우, 대출이 가능하다고 알리세요.
SQL Result가 '보안 규정상 조회가 어렵다' 는 결과로 나타나는 경우 답변을 거절하세요.'''),
('human','''
Question: {input}
SQL Query: {query}
SQL Result: {result}''')])

answer_chain = answer_prompt | llm | StrOutputParser()

chain = (
    RunnableParallel(input = RunnablePassthrough()).assign(query = query_chain | parse_sql).assign(result = execute_query).assign(answer= answer_chain)
)

chain.invoke("현재 연체 중인 도서와 연체료가 얼마인지 알려주세요. 누가 연체를 하나요?")

{'input': '현재 연체 중인 도서와 연체료가 얼마인지 알려주세요. 누가 연체를 하나요?',
 'query': 'SELECT\n    "books"."title" AS "도서명",\n    "members"."name" AS "연체자",\n    COALESCE(SUM("late_fees"."amount"), 0) AS "연체료"\nFROM "rentals"\nJOIN "book_copies"\n    ON "rentals"."copy_id" = "book_copies"."copy_id"\nJOIN "books"\n    ON "book_copies"."book_id" = "books"."book_id"\nJOIN "members"\n    ON "rentals"."member_id" = "members"."member_id"\nLEFT JOIN "late_fees"\n    ON "rentals"."rental_id" = "late_fees"."rental_id"\nWHERE "rentals"."status" = \'overdue\'\n  AND "rentals"."return_date" IS NULL\n  AND "rentals"."due_date" < date(\'now\')\nGROUP BY "rentals"."rental_id", "books"."title", "members"."name"\nORDER BY "rentals"."due_date"\nLIMIT 5',
 'result': "[('노인과 바다', '이하늘', 250.0), ('이상한 나라의 앨리스', '김민준', 300.0)]",
 'answer': '현재 연체 중인 도서는 **2권**입니다! 사건의 전모는 다음과 같습니다, 왓슨!\n\n- **『노인과 바다』** — 연체자: **이하늘**, 연체료: **250원**\n- **『이상한 나라의 앨리스』** — 연체자: **김민준**, 연체료: **300원**\n\n따라서 현재 확인된 **총 연체료는 550원**입니다. 사건 해결! (셜록 

In [ ]:
chain.invoke("삼성SDS RAG 심화과정 교재 100권 추가해줘")

{'input': '삼성SDS RAG 심화과정 교재 100권 추가해줘',
 'query': 'WITH RECURSIVE numbers(n) AS (\n    SELECT 1\n    UNION ALL\n    SELECT n + 1\n    FROM numbers\n    WHERE n < 100\n)\nINSERT INTO "books" ("title")\nSELECT \'삼성SDS RAG 심화과정 교재\'\nFROM numbers;',
 'result': '',
 'answer': '삼성SDS RAG 심화과정 교재 **100권 추가를 완료했습니다!** 이제 서가가 지식으로 꽉 차겠군요—책들이 서로 임베딩을 주고받으며 속삭일지도 모릅니다! 📚✨\n\n(셜록 홈즈 톤으로)'}

In [ ]:
chain.invoke("삼성SDS RAG 심화과정 교재 몇 권 있어?")

{'input': '삼성SDS RAG 심화과정 교재 몇 권 있어?',
 'query': 'SELECT COUNT(*) AS "교재_권수"\nFROM "books"\nWHERE "title" LIKE \'%삼성SDS%\'\n   OR "title" LIKE \'%RAG%\';',
 'result': '[(200,)]',
 'answer': '오호라! 도서관의 장부를 샅샅이 살펴본 결과, **삼성SDS RAG 심화과정 교재는 총 200권** 있소이다! 책의 산이로군, 산이야! (셜록 홈즈 톤으로)'}

In [ ]:
chain.invoke("삼성SDS RAG 심화과정 교재 다 삭제해줘.")

{'input': '삼성SDS RAG 심화과정 교재 다 삭제해줘.',
 'query': 'DELETE FROM "books"\nWHERE "title" = \'삼성SDS RAG 심화과정 교재\'',
 'result': '',
 'answer': '삼성SDS RAG 심화과정 교재 삭제 쿼리는 실행되었으나, 반환된 결과가 없어 **실제 삭제 건수는 확인할 수 없습니다**. 해당 제목의 교재가 존재했다면 삭제되었고, 없었다면 변경된 자료가 없습니다. (셜록 홈즈 톤으로)'}

테스트 질문
- 이메일이 haneul@example.com인 회원의 정보를 알려주세요.
- 해리 포터와 마법사의 돌을 지금 빌릴 수 있나요?
- 김민준 회원이 지금까지 빌린 책 목록을 보여주세요.
- 현재 연체 중인 도서와 연체료가 얼마인지 알려주세요.
- 판타지 장르에서 가장 인기 있는 책 3개를 추천해주세요.
- 최근 6개월간 연체 이력이 없는 회원을 찾아서 우수 회원으로 업그레이드 추천해주세요.

## Human(LLM) In the Loop 만들기

코드는 잘 구성되어 있지만, 위에서 진행했던 것처럼 위험한 쿼리를 실행하는 것은 사전에 방지할 필요가 있습니다.    

복잡한 로직도 가능하지만, 여기서는 중간 쿼리를 검증하여 Y/N으로 판독할 수 있는 단순한 모듈을 추가해 보겠습니다.    

   
질문과 쿼리를 보고, 해당 내용이 안전하지 않으면 이후 과정을 실행하지 않겠습니다.

In [ ]:
validation_prompt = """아래의 SQL 쿼리를 서버에서 실행하고자 합니다.
쿼리의 안전성을 평가하세요.
Approve하면 Y, 아니면 아무 키나 입력하세요.
---

Question = {input}

---
SQL Query = {query}
"""

def human_approval(msg):
    val_msg = validation_prompt.format(query = msg['query'], input = msg['input'])

    print(type(msg))
    resp = input(val_msg)

    if resp.upper() == 'Y':
        return execute_query

    else:
        return '보안 규정상 실행할 수 없음'



In [ ]:
sql_chain_with_validation = RunnableParallel(input = RunnablePassthrough()).assign(query =
                                                       query_chain | parse_sql).assign(
                                                           result = human_approval).assign(
                                                               answer = answer_chain)

sql_chain_with_validation.invoke("현재 연체 중인 도서와 연체료가 얼마인지 알려주세요.")

<class 'dict'>
아래의 SQL 쿼리를 서버에서 실행하고자 합니다.
쿼리의 안전성을 평가하세요.
Approve하면 Y, 아니면 아무 키나 입력하세요.
---

Question = 현재 연체 중인 도서와 연체료가 얼마인지 알려주세요.

---
SQL Query = SELECT
  "books"."title",
  "members"."name" AS "member_name",
  "rentals"."due_date",
  "late_fees"."amount" AS "late_fee"
FROM "rentals"
JOIN "book_copies" ON "rentals"."copy_id" = "book_copies"."copy_id"
JOIN "books" ON "book_copies"."book_id" = "books"."book_id"
JOIN "members" ON "rentals"."member_id" = "members"."member_id"
LEFT JOIN "late_fees" ON "rentals"."rental_id" = "late_fees"."rental_id"
WHERE "rentals"."status" = 'overdue'
  AND "rentals"."return_date" IS NULL
ORDER BY "late_fees"."amount" DESC
LIMIT 5
0


{'input': '현재 연체 중인 도서와 연체료가 얼마인지 알려주세요.',
 'query': 'SELECT\n  "books"."title",\n  "members"."name" AS "member_name",\n  "rentals"."due_date",\n  "late_fees"."amount" AS "late_fee"\nFROM "rentals"\nJOIN "book_copies" ON "rentals"."copy_id" = "book_copies"."copy_id"\nJOIN "books" ON "book_copies"."book_id" = "books"."book_id"\nJOIN "members" ON "rentals"."member_id" = "members"."member_id"\nLEFT JOIN "late_fees" ON "rentals"."rental_id" = "late_fees"."rental_id"\nWHERE "rentals"."status" = \'overdue\'\n  AND "rentals"."return_date" IS NULL\nORDER BY "late_fees"."amount" DESC\nLIMIT 5',
 'result': '보안 규정상 실행할 수 없음',
 'answer': '죄송하지만, 보안 규정으로 인해 현재 연체 중인 도서와 연체료 정보를 조회하거나 안내할 수 없습니다. 도서관 직원에게 직접 문의해 주십시오. (셜록 홈즈 톤으로)'}

# [실습] LLM으로 validation 검증 체인 추가하기   

위의 `sql_chain_with_validation` 구조를 수정하여,    
Human Approval 대신 LLM이 이를 검증하는 체인을 추가하세요.   
다음은 예시 프롬프트입니다.

```python   
아래의 SQL 쿼리를 서버에서 실행하고자 합니다.
SQL 쿼리는 새로운 데이터를 추가하거나, 기존의 값을 변경하거나 삭제해서는 안 됩니다.
쿼리의 안전성에 대해 먼저 30자 이내로 설명하세요.
안전하면 "분류 결과: Y", 아니면 "분류 결과: N"을 출력하세요.

SQL Query = {query}
```

In [ ]:
llm_validation_prompt = ChatPromptTemplate([
 ('system','''
아래의 SQL 쿼리를 서버에서 실행하고자 합니다.
SQL 쿼리는 새로운 데이터를 추가하거나, 기존의 값을 변경하거나 삭제해서는 안 됩니다.
쿼리의 안전성에 대해 먼저 30자 이내로 설명하세요.
안전하면 "분류 결과: Y", 아니면 "분류 결과: N"을 출력하세요.'''),
 ('human','''SQL Query = {query}''')])
# 분류 결과를 print로 출력

def llm_approval(msg):
    approval_chain = llm_validation_prompt | llm | StrOutputParser()

    result = approval_chain.invoke(msg)
    print(result)
    if '분류 결과: Y' in result:
        return execute_query
    else:
        return '보안 규정상 실행 불가'

In [ ]:
sql_chain_with_validation = RunnableParallel(input = RunnablePassthrough()).assign(query =
                                                       query_chain | parse_sql).assign(
                                                           result = llm_approval).assign(
                                                               answer = answer_chain)

sql_chain_with_validation.invoke("삼성SDS RAG 심화 과정 교재 200권 추가해줘")

데이터를 추가하는 INSERT 문이므로 안전하지 않습니다.
분류 결과: N


{'input': '삼성SDS RAG 심화 과정 교재 200권 추가해줘',
 'query': 'INSERT INTO "book_copies" ("copy_id", "book_id", "barcode", "condition", "location", "acquisition_date", "status")\nWITH RECURSIVE "seq"("n") AS (\n    SELECT 1\n    UNION ALL\n    SELECT "n" + 1\n    FROM "seq"\n    WHERE "n" < 200\n)\nSELECT\n    COALESCE((SELECT MAX("copy_id") FROM "book_copies"), 0) + "n",\n    (SELECT "book_id" FROM "books" WHERE "title" = \'삼성SDS RAG 심화 과정 교재\' LIMIT 1),\n    \'SDS-RAG-\' || printf(\'%03d\', "n"),\n    \'good\',\n    \'미지정\',\n    date(\'now\'),\n    \'available\'\nFROM "seq"\nWHERE EXISTS (\n    SELECT 1\n    FROM "books"\n    WHERE "title" = \'삼성SDS RAG 심화 과정 교재\'\n);',
 'result': '보안 규정상 실행 불가',
 'answer': '요청하신 **「삼성SDS RAG 심화 과정 교재」 200권 추가 작업은 보안 규정상 실행할 수 없습니다.** 따라서 현재 재고에는 반영되지 않았습니다. (셜록 홈즈 톤으로)'}